In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)

plt.rcParams["figure.figsize"] = (12,6)
sns.set_theme(style="whitegrid")

# Case The News - Palavritas

## Objetivo

Responder à seguinte pergunta:

> Quais variáveis mais se correlacionam com o usuário estar ativo no D30 e voltar a jogar no dia seguinte?

Durante a análise serão exploradas:

- Horário de jogo
- Palavra do dia
- Perfil do usuário
- Newsletter
- Food Delivery
- Correlações entre variáveis

Ao final serão propostas hipóteses de negócio e experimentos para validação.

In [ ]:
df = pd.read_parquet("data/df_cleaned.parquet")

df.head()

# 1. Conhecendo os Dados

In [ ]:
print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
print("Duplicidades:", df.duplicated().sum())

# 2. Distribuição das Variáveis

In [ ]:
px.histogram(
    df,
    x="age_range",
    color="played_next_day",
    barmode="group",
    title="Faixa Etária x Retorno D+1"
).show()

In [ ]:
px.histogram(
    df,
    x="salary_range",
    color="played_next_day",
    barmode="group",
    title="Faixa Salarial x Retorno D+1"
).show()

In [ ]:
px.histogram(
    df,
    x="primary_device",
    color="played_next_day",
    title="Device x Retorno D+1"
).show()

In [ ]:
px.histogram(
    df,
    x="sector",
    color="played_next_day",
    title="Setor x Retorno D+1"
).show()

# 3. Horário de Jogo

In [ ]:
px.histogram(
    df,
    x="session_hour",
    color="played_next_day",
    histnorm="percent",
    barmode="group",
    nbins=24,
    title="Distribuição das sessões ao longo do dia"
).show()

In [ ]:
px.box(
    df,
    x="played_next_day",
    y="time_to_complete_sec",
    color="played_next_day",
    title="Tempo para finalizar o jogo x Retorno"
).show()

### Hipótese

**Hipótese**

"Acredito que usuários que jogam à noite apresentam maior retenção porque esse horário faz parte da rotina diária."

**Ação**

Enviar notificações próximas ao horário habitual de jogo.

**Critério de sucesso**

"Saberei que funcionou quando a retenção D+1 e D30 aumentarem em relação ao grupo controle."

In [ ]:
word_stats = (
    df.groupby("word")
      .agg(
          taxa_retencao=("played_next_day","mean"),
          tentativas=("attempt_number","mean"),
          volume=("user_id","count")
      )
      .reset_index()
)

In [ ]:
fig = px.scatter(
    word_stats,
    x="tentativas",
    y="taxa_retencao",
    size="volume",
    color="taxa_retencao",
    hover_name="word",
    title="Palavra do Dia: Tentativas x Retenção",
    color_continuous_scale="RdYlGn"
)

fig.add_hline(
    y=word_stats["taxa_retencao"].mean(),
    line_dash="dot"
)

fig.show()

### Hipótese

**Hipótese**

"Acredito que palavras excessivamente difíceis reduzem a retenção porque aumentam a frustração."

**Ação**

Balancear a dificuldade das palavras e testar dicas opcionais.

**Critério de sucesso**

"Saberei que funcionou quando as palavras difíceis apresentarem aumento da retenção D+1."

In [ ]:
def conversion_plot(feature, title):

    agg = (
        df.groupby(feature)["played_next_day"]
        .agg(["mean","count"])
        .reset_index()
        .sort_values("mean")
    )

    fig = px.bar(
        agg,
        y=feature,
        x="mean",
        orientation="h",
        text="mean",
        color="mean",
        color_continuous_scale="Blues",
        title=title
    )

    fig.update_traces(texttemplate="%{text:.1%}")

    fig.update_layout(coloraxis_showscale=False)

    fig.show()

In [ ]:
conversion_plot("salary_range","Retenção por Faixa Salarial")

In [ ]:
conversion_plot("sector","Retenção por Setor")

In [ ]:
conversion_plot("primary_device","Retenção por Device")

In [ ]:
conversion_plot("age_range","Retenção por Faixa Etária")

### Hipótese

**Hipótese**

"Acredito que determinados perfis apresentam maior retenção porque o jogo se encaixa melhor em seus hábitos."

**Ação**

Criar campanhas segmentadas para os perfis de menor retenção.

**Critério de sucesso**

"Saberei que funcionou quando a diferença entre segmentos diminuir."

# 6. Newsletter

In [ ]:
conversion_plot(
    "newsletter_open_before_game",
    "Newsletter aberta antes do jogo"
)

In [ ]:
conversion_plot(
    "newsletter_subscriber",
    "Assinante da Newsletter"
)

### Hipótese

**Hipótese**

"Acredito que usuários que abrem a newsletter retornam mais porque já demonstram maior engajamento com o produto."

**Ação**

Adicionar CTA do Palavritas na newsletter e realizar testes A/B.

**Critério de sucesso**

"Saberei que funcionou quando o grupo exposto apresentar maior retenção D+1 e D30."

# 7. Food Delivery

In [ ]:
conversion_plot(
    "food_delivery_freq_week",
    "Frequência de Food Delivery"
)

### Hipótese

**Hipótese**

"Acredito que usuários que utilizam delivery com frequência possuem momentos recorrentes de espera, favorecendo sessões de jogo."

**Ação**

Criar campanhas em horários de refeição e avaliar parcerias com cupons.

**Critério de sucesso**

"Saberei que funcionou quando esse segmento apresentar aumento da retenção."

# 8. Correlação

In [ ]:
bool_cols = [
    "played_next_day",
    "active_d30",
    "newsletter_open_before_game",
    "newsletter_subscriber",
    "orders_food_delivery",
    "plays_other_word_games"
]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

In [ ]:
corr = df.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(16,10))

sns.heatmap(
    corr,
    annot=True,
    cmap="RdBu_r",
    center=0,
    fmt=".2f"
)

plt.title("Matriz de Correlação")
plt.show()

In [ ]:
corr["played_next_day"].sort_values(ascending=False)

In [ ]:
corr["active_d30"].sort_values(ascending=False)

# 9. Principais Achados

## Horário de jogo

Foi observada associação entre o horário em que o usuário joga e sua probabilidade de retorno, indicando que hábitos de rotina podem influenciar a retenção.

---

## Palavra do dia

Palavras que exigem mais tentativas tendem a apresentar menor retenção, sugerindo que dificuldade excessiva pode gerar frustração.

---

## Perfil do usuário

Diferenças entre faixa salarial, setor de atuação, faixa etária e dispositivo indicam que determinados segmentos apresentam maior engajamento.

---

## Newsletter

Usuários que interagem com a newsletter apresentam maior probabilidade de retorno, sugerindo maior envolvimento com o ecossistema do produto.

---

## Food Delivery

A frequência de pedidos de delivery pode representar um indicador de hábitos de consumo associados ao momento de uso do jogo, sendo uma oportunidade para campanhas segmentadas.

# 10. Conclusão

A análise exploratória identificou variáveis associadas ao retorno no dia seguinte e à retenção em D30.

Como correlação não implica causalidade, as hipóteses levantadas devem ser validadas por meio de experimentos controlados (A/B Tests).

As principais oportunidades identificadas foram:

- Personalização por horário de jogo.
- Balanceamento da dificuldade das palavras.
- Comunicação segmentada por perfil.
- Integração da newsletter com o Palavritas.
- Campanhas direcionadas para usuários com maior frequência de food delivery.

Essas iniciativas possuem potencial para aumentar tanto o retorno no dia seguinte quanto a retenção de longo prazo (D30).